# Classification Model Comparison

**Student name:** ____________________  
**Student ID:** ____________________

This notebook implements five classifiers on one UCI dataset and evaluates all six required metrics on one shared held-out test set.

**Execution note:** Outputs were generated and checked in a local Python environment.

## 1. Dataset selection

**Breast Cancer Wisconsin (Diagnostic)** — binary classification, **569 instances**, **30 numeric input features**. This satisfies the minimum of 500 instances and 12 features. The target and patient ID are not counted as input features.

Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).  
Citation: Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). *Breast Cancer Wisconsin (Diagnostic)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B. License: CC BY 4.0.

The 30 features measure the mean, standard error, and worst value of ten cell nucleus characteristics: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, and fractal dimension. There are 212 malignant and 357 benign observations and no missing values.

We use scikit-learn's bundled copy of this UCI dataset, so no internet access or Kaggle credentials are required. The original scikit-learn labels are remapped to **malignant = 1** and **benign = 0**. Precision, recall, F1, and AUC therefore refer to detecting malignant cases.

## 2. Imports

Dependencies: Python 3, NumPy, pandas, and scikit-learn. If an import fails, install the missing packages in that notebook environment using `%pip install numpy pandas scikit-learn`, then restart the kernel. Installation is unnecessary if the imports succeed.


In [1]:
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix
)

RANDOM_STATE = 42
print("Python:", platform.python_version())
print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)


Python: 3.12.14
NumPy: 2.3.5 | pandas: 2.2.3 | scikit-learn: 1.8.0


## 3. Load and inspect data

No identifiers or target values are included in the feature matrix. The assertions check the assignment's dataset size requirements and confirm complete, finite inputs.


In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series((data.target == 0).astype(int), name="malignant")

assert X.shape[0] >= 500 and X.shape[1] >= 12
assert X.shape[0] == len(y)
assert not X.isna().any().any()
assert np.isfinite(X.to_numpy()).all()
assert set(y.unique()) == {0, 1}

print("Dataset: UCI Breast Cancer Wisconsin (Diagnostic)")
print("Instances:", X.shape[0], "| Features:", X.shape[1])
print("Class counts:", y.map({0: "Benign", 1: "Malignant"}).value_counts().to_dict())
print("Missing feature values:", int(X.isna().sum().sum()))
print("\nFeature names:")
print(", ".join(X.columns))


Dataset: UCI Breast Cancer Wisconsin (Diagnostic)
Instances: 569 | Features: 30
Class counts: {'Benign': 357, 'Malignant': 212}
Missing feature values: 0

Feature names:
mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness, mean concavity, mean concave points, mean symmetry, mean fractal dimension, radius error, texture error, perimeter error, area error, smoothness error, compactness error, concavity error, concave points error, symmetry error, fractal dimension error, worst radius, worst texture, worst perimeter, worst area, worst smoothness, worst compactness, worst concavity, worst concave points, worst symmetry, worst fractal dimension


## 4. Shared stratified train/test split

Use 80% of the observations for training and 20% for testing: 455 training and 114 testing observations. Stratification approximately preserves the class proportions. Every model receives exactly the same split, with `random_state=42` for reproducibility.

Scaling is learned inside each pipeline from training data only. It is particularly useful for Logistic Regression, KNN, and SVM. Gaussian NB and the Decision Tree use the original feature scales. There is no hyperparameter search or test-set-driven tuning.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Training positives:", int(y_train.sum()), "| Test positives:", int(y_test.sum()))


Training shape: (455, 30) | Test shape: (114, 30)
Training positives: 170 | Test positives: 42


## 5. Define the five models

1. **Logistic Regression:** L2 regularization, C=1, lbfgs solver, 5000 maximum iterations; standardized inputs.
2. **Decision Tree:** Gini criterion, unrestricted depth, fixed random state.
3. **K-Nearest Neighbors:** k=5, uniform weights, Euclidean distance; standardized inputs.
4. **Gaussian Naive Bayes:** Gaussian likelihood for continuous features, var_smoothing=1e-9.
5. **Support Vector Machine:** RBF kernel, C=1, gamma='scale'; standardized inputs.

These are baseline configurations, not claims of optimized performance.


In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0, solver="lbfgs", max_iter=5000, random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE
    ),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5, weights="uniform", metric="minkowski", p=2
        ))
    ]),
    "Gaussian Naive Bayes": GaussianNB(var_smoothing=1e-9),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(C=1.0, kernel="rbf", gamma="scale"))
    ])
}


## 6. Train and calculate the six required metrics

- **Accuracy:** fraction of all predictions that are correct.
- **ROC AUC:** ranking performance across decision thresholds; calculated from malignant-class probabilities or SVM decision scores, not predicted class labels.
- **Precision:** TP / (TP + FP).
- **Recall:** TP / (TP + FN).
- **F1:** harmonic mean of precision and recall.
- **Matthews correlation coefficient (MCC):** (TP × TN − FP × FN) / sqrt((TP + FP)(TP + FN)(TN + FP)(TN + FN)).

TP means a correctly predicted malignant case. All scores except MCC range from 0 to 1; MCC ranges from −1 to +1. Precision, recall, and F1 use the binary positive class 1. `zero_division=0` handles undefined precision or recall. Scores are kept at full precision and only rounded for display.


In [5]:
rows = []
confusion_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        # Explicitly locate the malignant (1) probability column.
        positive_index = int(np.flatnonzero(model.classes_ == 1)[0])
        y_score = model.predict_proba(X_test)[:, positive_index]
    else:
        # Binary SVC scores increase toward classes_[1], which is malignant.
        assert list(model.classes_) == [0, 1]
        y_score = model.decision_function(X_test)

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    confusion_rows.append({"Model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp})

results = pd.DataFrame(rows).set_index("Model")
confusions = pd.DataFrame(confusion_rows).set_index("Model")
assert results.shape == (5, 6)
assert np.isfinite(results.to_numpy()).all()
print("All five models trained and evaluated successfully.")


All five models trained and evaluated successfully.


## Save trained models as pickle files

Run this cell after training to save all five fitted models in `models/` under the notebook working directory. The Logistic Regression, KNN and SVM pipelines include their fitted scalers. Load these files only from trusted sources, using the Python and package versions in `validated_environment.txt`.


In [6]:
import pickle
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

filenames = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "K-Nearest Neighbors": "knn.pkl",
    "Gaussian Naive Bayes": "naive_bayes.pkl",
    "Support Vector Machine": "svm.pkl",
}

for name, model in models.items():
    path = model_dir / filenames[name]
    with path.open("wb") as file:
        pickle.dump(model, file, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved: {path}")


Saved: models/logistic_regression.pkl
Saved: models/decision_tree.pkl
Saved: models/knn.pkl
Saved: models/naive_bayes.pkl
Saved: models/svm.pkl


## 7. Confusion matrices and interpretation

TN = benign correctly classified; FP = benign predicted malignant; FN = malignant predicted benign; TP = malignant correctly classified. These counts help explain precision and recall.


In [7]:
print(confusions.to_string())
print("\nHighest observed test accuracy:", results["Accuracy"].idxmax())
print("Highest observed test AUC:", results["AUC"].idxmax())

results.to_csv("classification_results.csv")
print("\nSaved classification_results.csv in the notebook working directory.")


                        TN  FP  FN  TP
Model                                 
Logistic Regression     71   1   3  39
Decision Tree           68   4   4  38
K-Nearest Neighbors     71   1   4  38
Gaussian Naive Bayes    72   0   7  35
Support Vector Machine  72   0   3  39

Highest observed test accuracy: Support Vector Machine
Highest observed test AUC: Logistic Regression

Saved classification_results.csv in the notebook working directory.


A single 114-observation test split gives a limited estimate: one additional error changes accuracy by approximately 0.88 percentage points. The best score on this split does not establish that a model is universally best. Correlated features can weaken Gaussian NB's conditional independence assumption; tree performance can be sensitive to the training split. Repeated stratified cross-validation would support a stronger comparison but is beyond the required baseline evaluation.

## 8. Final comparison

The final cell displays all five models and six requested metrics. The results are from a local run; no BITS Virtual Lab execution or screenshot is claimed.


In [8]:
print("CLASSIFICATION MODEL COMPARISON")
print("UCI Breast Cancer Wisconsin (Diagnostic) | 569 instances | 30 features")
print("Shared stratified 80:20 split | Train: 455 | Test: 114 | Seed: 42")
print("Positive class: malignant (1) | AUC = ROC AUC | Scores rounded to 4 decimals")
print()
print(results.to_string(float_format=lambda value: f"{value:.4f}"))
print("\nPython", platform.python_version(), "| scikit-learn", sklearn.__version__)


CLASSIFICATION MODEL COMPARISON
UCI Breast Cancer Wisconsin (Diagnostic) | 569 instances | 30 features
Shared stratified 80:20 split | Train: 455 | Test: 114 | Seed: 42
Positive class: malignant (1) | AUC = ROC AUC | Scores rounded to 4 decimals

                        Accuracy    AUC  Precision  Recall     F1    MCC
Model                                                                   
Logistic Regression       0.9649 0.9960     0.9750  0.9286 0.9512 0.9245
Decision Tree             0.9298 0.9246     0.9048  0.9048 0.9048 0.8492
K-Nearest Neighbors       0.9561 0.9823     0.9744  0.9048 0.9383 0.9058
Gaussian Naive Bayes      0.9386 0.9934     1.0000  0.8333 0.9091 0.8715
Support Vector Machine    0.9737 0.9947     1.0000  0.9286 0.9630 0.9442

Python 3.12.14 | scikit-learn 1.8.0
